## Learning Objectives






##  Import Libraries and Load Data

We will use the same titanic dataset and model preparation as the previous chapter for this chapter


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# For statsmodels implementation
import statsmodels.formula.api as smf

# For sklearn implementation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the Titanic dataset
titanic = sns.load_dataset('titanic')

print(f"Dataset shape: {titanic.shape}")
print(f"\nFirst few rows:")
titanic.head()

Dataset shape: (891, 15)

First few rows:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


##  Data Preparation

### Imputing missing values


Like linear regression, logistic regression does not tolerate missing values

We'll focus on:
- **Target variable:** `survived` (0 = died, 1 = survived)
- **Predictors:** `age`, `fare`, `sex`, `pclass`

Let's check for missing values in these columns


In [3]:
# Check for missing values in our columns of interest
columns_of_interest = ['survived', 'age', 'fare', 'sex', 'pclass']
print("Missing values in columns of interest:")
print(titanic[columns_of_interest].isnull().sum())

Missing values in columns of interest:
survived      0
age         177
fare          0
sex           0
pclass        0
dtype: int64


There are 177 missing values in Age, which is a key predictor. Dropping all rows with missing values would result in a substantial loss of useful information. A simple and reasonable approach is group-wise median imputation: fill in missing ages using the median Age within each combination of `Sex` and `Pclass`.

In [4]:
# impute median age for each combination of sex and pclass
titanic['age'] = titanic.groupby(['sex', 'pclass'])['age'].transform(lambda x: x.fillna(x.median()))

Let's check again

In [5]:
columns_of_interest = ['survived', 'age', 'fare', 'sex', 'pclass']
print("Missing values in columns of interest:")
print(titanic[columns_of_interest].isnull().sum())

Missing values in columns of interest:
survived    0
age         0
fare        0
sex         0
pclass      0
dtype: int64


### Check Target Distribution

Let’s examine the distribution of the target variable to determine whether the dataset is balanced.

In [6]:
# Quick exploratory analysis
print("Target variable distribution:")
print(titanic['survived'].value_counts())

Target variable distribution:
survived
0    549
1    342
Name: count, dtype: int64


## Stratified Splitting 

The dataset shows moderate class imbalance (about 62% non-survived vs 38% survived). While not extreme, this imbalance suggests we should look beyond accuracy and also consider metrics such as precision, recall, and ROC-AUC.

For such dataset, we need to do **stratified splitting** to make sure the proportion of classes is preserved in both train and test sets.



In [7]:
# Split into train and test sets
train_df, test_df = train_test_split(titanic, test_size=0.2, random_state=42, stratify=titanic['survived'])

In [8]:
# Let's compare random splitting vs stratified splitting
print("="*70)
print("COMPARING RANDOM vs STRATIFIED SPLITTING")
print("="*70)

# Original class distribution
print("\n1. ORIGINAL DATASET CLASS DISTRIBUTION:")
print("-"*70)
original_counts = titanic['survived'].value_counts()
original_proportions = titanic['survived'].value_counts(normalize=True)
print(f"  Died (0):     {original_counts[0]:4d} samples ({original_proportions[0]:.2%})")
print(f"  Survived (1): {original_counts[1]:4d} samples ({original_proportions[1]:.2%})")
print(f"  Total:        {len(titanic):4d} samples")

# Random splitting (WITHOUT stratify)
print("\n2. RANDOM SPLITTING (without stratify):")
print("-"*70)
train_random, test_random = train_test_split(titanic, test_size=0.2, random_state=42)

train_random_counts = train_random['survived'].value_counts()
train_random_props = train_random['survived'].value_counts(normalize=True)
test_random_counts = test_random['survived'].value_counts()
test_random_props = test_random['survived'].value_counts(normalize=True)

print(f"  Train Set:")
print(f"    Died (0):     {train_random_counts[0]:4d} samples ({train_random_props[0]:.2%})")
print(f"    Survived (1): {train_random_counts[1]:4d} samples ({train_random_props[1]:.2%})")
print(f"  Test Set:")
print(f"    Died (0):     {test_random_counts[0]:4d} samples ({test_random_props[0]:.2%})")
print(f"    Survived (1): {test_random_counts[1]:4d} samples ({test_random_props[1]:.2%})")

# Calculate deviation from original proportions
train_random_deviation = abs(train_random_props[1] - original_proportions[1])
test_random_deviation = abs(test_random_props[1] - original_proportions[1])
print(f"\n  Deviation from original proportions:")
print(f"    Train: {train_random_deviation:.3%}")
print(f"    Test:  {test_random_deviation:.3%}")

# Stratified splitting (WITH stratify)
print("\n3. STRATIFIED SPLITTING (with stratify):")
print("-"*70)
train_stratified, test_stratified = train_test_split(titanic, test_size=0.2, random_state=42, 
                                                      stratify=titanic['survived'])

train_strat_counts = train_stratified['survived'].value_counts()
train_strat_props = train_stratified['survived'].value_counts(normalize=True)
test_strat_counts = test_stratified['survived'].value_counts()
test_strat_props = test_stratified['survived'].value_counts(normalize=True)

print(f"  Train Set:")
print(f"    Died (0):     {train_strat_counts[0]:4d} samples ({train_strat_props[0]:.2%})")
print(f"    Survived (1): {train_strat_counts[1]:4d} samples ({train_strat_props[1]:.2%})")
print(f"  Test Set:")
print(f"    Died (0):     {test_strat_counts[0]:4d} samples ({test_strat_props[0]:.2%})")
print(f"    Survived (1): {test_strat_counts[1]:4d} samples ({test_strat_props[1]:.2%})")

# Calculate deviation from original proportions
train_strat_deviation = abs(train_strat_props[1] - original_proportions[1])
test_strat_deviation = abs(test_strat_props[1] - original_proportions[1])
print(f"\n  Deviation from original proportions:")
print(f"    Train: {train_strat_deviation:.3%}")
print(f"    Test:  {test_strat_deviation:.3%}")

print("\n" + "="*70)
print("KEY OBSERVATIONS:")
print("="*70)
print("✓ Stratified splitting maintains the SAME class proportions in train/test")
print("✓ Random splitting may create different distributions by chance")
print("✓ For imbalanced datasets, stratified splitting ensures:")
print("  - Fair representation of minority class in both sets")
print("  - More reliable model evaluation")
print("  - Consistent results across different train/test splits")
print("="*70)

COMPARING RANDOM vs STRATIFIED SPLITTING

1. ORIGINAL DATASET CLASS DISTRIBUTION:
----------------------------------------------------------------------
  Died (0):      549 samples (61.62%)
  Survived (1):  342 samples (38.38%)
  Total:         891 samples

2. RANDOM SPLITTING (without stratify):
----------------------------------------------------------------------
  Train Set:
    Died (0):      444 samples (62.36%)
    Survived (1):  268 samples (37.64%)
  Test Set:
    Died (0):      105 samples (58.66%)
    Survived (1):   74 samples (41.34%)

  Deviation from original proportions:
    Train: 0.743%
    Test:  2.957%

3. STRATIFIED SPLITTING (with stratify):
----------------------------------------------------------------------
  Train Set:
    Died (0):      439 samples (61.66%)
    Survived (1):  273 samples (38.34%)
  Test Set:
    Died (0):      110 samples (61.45%)
    Survived (1):   69 samples (38.55%)

  Deviation from original proportions:
    Train: 0.041%
    Test:  0.

##  Scikit-learn Implementation

Since `sklearn` is optimized for **prediction and machine learning workflows**. we will stick with `sklearn`

### Pipeline Implementation

**Why use Pipelines?**

In the previous chapter, we've manually created dummy variables using `pd.get_dummies()`. While this works, sklearn's **Pipeline** approach offers several advantages:

1. **Automated preprocessing**: Handles categorical encoding automatically
2. **Prevents data leakage**: Ensures transformations are fit only on training data
3. **Cleaner code**: Combines preprocessing and modeling in one object
4. **Production-ready**: Easier to deploy and maintain
5. **Reproducibility**: All steps are encapsulated in one pipeline

**Key components:**
- `ColumnTransformer`: Applies different transformations to different columns
- `OneHotEncoder`: Converts categorical variables to dummy variables
- `Pipeline`: Chains preprocessing and model fitting together

Let's implement the same logistic regression model using a pipeline (without regularization to match statsmodels):

In [ ]:
# Import required components for pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Define which columns are categorical and which are numeric
categorical_features = ['sex']
numeric_features = ['age', 'fare', 'pclass']

# Create the column transformer
# OneHotEncoder with drop='first' mimics the drop_first=True in pd.get_dummies
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),  # Keep numeric features as-is
        ('cat', OneHotEncoder(drop='first'), categorical_features)  # Encode categorical
    ])

# Create the pipeline: preprocessing + logistic regression
pipeline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(C=np.inf, random_state=42))
])

# Fit the pipeline on training data
# Note: We pass the original dataframe with categorical variables
X_train_raw = train_df[['age', 'fare', 'sex', 'pclass']]
X_test_raw = test_df[['age', 'fare', 'sex', 'pclass']]

pipeline_model.fit(X_train_raw, y_train)

print("Pipeline model fitted successfully!")
print("\nPipeline structure:")
print(pipeline_model)

Pipeline model fitted successfully!

Pipeline structure:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['age', 'fare', 'pclass']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['sex'])])),
                ('classifier', LogisticRegression(C=inf, random_state=42))])


In [ ]:
# Extract the trained logistic regression model from the pipeline
pipeline_logit = pipeline_model.named_steps['classifier']

# Get feature names after transformation
feature_names = numeric_features + ['sex_male']  # OneHotEncoder creates sex_male (drops sex_female)

print("Pipeline Logistic Regression Coefficients (C=np.inf):")
print(f"\nCoefficients:")
for feature, coef in zip(feature_names, pipeline_logit.coef_[0]):
    print(f"  {feature:20s}: {coef:8.4f}")
print(f"\nIntercept: {pipeline_logit.intercept_[0]:.4f}")

# Make predictions
pipeline_train_pred = pipeline_model.predict(X_train_raw)
pipeline_test_pred = pipeline_model.predict(X_test_raw)

# Calculate accuracy
pipeline_train_accuracy = accuracy_score(y_train, pipeline_train_pred)
pipeline_test_accuracy = accuracy_score(y_test, pipeline_test_pred)

print(f"\nPipeline Model Performance:")
print(f"  Training Accuracy:   {pipeline_train_accuracy:.4f} ({pipeline_train_accuracy:.2%})")
print(f"  Test Accuracy:       {pipeline_test_accuracy:.4f} ({pipeline_test_accuracy:.2%})")

Pipeline Logistic Regression Coefficients (C=np.inf):

Coefficients:
  age                 :  -0.0400
  fare                :   0.0007
  pclass              :  -1.2733
  sex_male            :  -2.6032

Intercept: 5.1031

Pipeline Model Performance:
  Training Accuracy:   0.7992 (79.92%)
  Test Accuracy:       0.7821 (78.21%)
